# Training and Evaluating Machine Learning Models with cuML

cuML is NVIDIA's GPU-accelerated machine learning library that implements popular ML algorithms with CUDA optimization. It
provides scikit-learn-like APIs while leveraging GPU acceleration to deliver significant speedups compared to CPU-based
implementations. cuML is part of the RAPIDS suite of open-source software libraries.

This notebook explores a classification and a regression machine learning estimator in cuML, demonstrating how to train them and evaluate them with built-in metrics functions. All of the models are trained on synthetic data, generated by cuML's dataset utilities.

**Note:**

cuML supports several estimators that you can find in the [cuML API documentation](https://docs.rapids.ai/api/cuml/stable/api/#api-reference), if you are particularly interested in clustering, there is extra material on UMAP and DBSCAN in the `extras` directory.


## Classification

### Random Forest Classification and Accuracy metrics

The Random Forest classification algorithm builds several decision trees, and aggregates each of their outputs to
make a prediction. For more information on cuML's implementation of the Random Forest Classification model please refer
to: [Random Forest Classifier API Documentation](https://docs.rapids.ai/api/cuml/stable/api.html#cuml.ensemble.RandomForestClassifier)

Accuracy score is the ratio of correct predictions to the total number of predictions. It is used to measure the performance
of classification models. For more information on the accuracy score metric please refer to [Wikipedia's article on Accuracy and Precision](https://en.wikipedia.org/wiki/Accuracy_and_precision).

For more information on cuML's implementation of accuracy score metrics please refer to the [cuML Accuracy Score API Documentation](https://docs.rapids.ai/api/cuml/stable/api.html#cuml.metrics.accuracy.accuracy_score).

The cell below shows an end to end pipeline of the Random Forest Classification model. Here the dataset was generated by
using scikit-learn's make_classification dataset. The generated dataset was used to train and run predict on the model.
Random forest's performance is evaluated and then compared between the values obtained from the cuML and scikit-learn accuracy metrics.

In [1]:
import cuml
from cupy import asnumpy
from joblib import dump, load

from cuml.datasets.classification import make_classification
from cuml.model_selection import train_test_split
from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.metrics import accuracy_score

In [2]:
%%time

# synthetic dataset dimensions
n_samples = 100_000
n_features = 10
n_classes = 2

# random forest depth and size
n_estimators = 25
max_depth = 10

# generate synthetic data [ binary classification task ]
X, y = make_classification(
    n_classes=n_classes,
    n_features=n_features,
    n_samples=n_samples,
)

X_train, X_test, y_train, y_test = train_test_split(X, y)

model = cuRF(
    max_depth=max_depth,
    n_estimators=n_estimators,
)

trained_RF = model.fit(X_train, y_train)

predictions = model.predict(X_test)

cu_score = cuml.metrics.accuracy_score(y_test, predictions)
sk_score = accuracy_score(asnumpy(y_test), asnumpy(predictions))

print(" cuml accuracy: ", cu_score)
print(" sklearn accuracy : ", sk_score)

 cuml accuracy:  0.9294
 sklearn accuracy :  0.9294
CPU times: user 23 s, sys: 876 ms, total: 23.9 s
Wall time: 31.3 s


In [ ]:
# if you want to save the model
dump(trained_RF, "RF.model")

# to reload the model uncomment the line below
# loaded_model = load('RF.model')

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
#sklearn cpu comparison
!python /content/drive/MyDrive/rapidsai-community/scripts/sklearn_RF.py

sklearn accuracy: 0.94752
Total elapsed time for RF: 7.6425 seconds


## Regression

### Linear regression and  R^2 score
Linear Regression is a simple machine learning model where the response `y` is modeled by a linear combination of the
predictors in `X`.

`R^2` score is also known as the coefficient of determination. It is used as a metric for scoring regression models. It
scores the output of the model based on the proportion of total variation of the model. For more information on the `R^2`
score metrics please refer to [Wikipedia page on Coefficient of determination](https://en.wikipedia.org/wiki/Coefficient_of_determination)

For more information on cuML's implementation of the `r2` score metrics please refer to [cuML R2 Score Documentation](https://docs.rapids.ai/api/cuml/stable/api.html)

The cell below uses the Linear Regression model to compare the results between cuML and scikit-learn trustworthiness metric. For more
information on cuML's implementation of the Linear Regression model please refer to [cuML Linear Regression Documentation](https://docs.rapids.ai/api/cuml/stable/api.html)

In [6]:
from cuml.datasets import make_regression
from cuml.linear_model import LinearRegression as cuLR
from sklearn.metrics import r2_score

In [7]:
%%time

n_samples = 2**10
n_features = 100
n_info = 70

X_reg, y_reg = make_regression(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=n_info,
)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, train_size=0.8,
)

cuml_reg_model = cuLR(fit_intercept=True, normalize=True, algorithm="eig")

trained_LR = cuml_reg_model.fit(X_reg_train, y_reg_train)
cu_preds = trained_LR.predict(X_reg_test)

cu_r2 = cuml.metrics.r2_score(y_reg_test, cu_preds)
sk_r2 = r2_score(asnumpy(y_reg_test), asnumpy(cu_preds))

print(f"cuml's r2 score : {cu_r2}")
print(f"sklearn's r2 score : {sk_r2}")

cuml's r2 score : 1.0
sklearn's r2 score : 1.0
CPU times: user 5.25 s, sys: 209 ms, total: 5.46 s
Wall time: 9.39 s


In [ ]:
# save and reload
# dump(trained_LR, "LR.model")

# to reload the model uncomment the line below
# loaded_model = load('LR.model')

# Example integration with an existing workflow
Chances are that you already have an existing workflow that uses `scikit-learn`. You may even have custom transformers implemented for data preprocessing.

This example takes a pipeline that uses [skrub](https://skrub-data.org/stable/index.html), an open-source package that aims at bridging the gap between tabular data sources and machine-learning models.

Here we will show how different tools that leverage the `scikit-learn` API can be combined in a pipeline.
## Easy learning on a dataframe

Let's first retrieve the dataset, using one of the downloaders from the `skrub.datasets` module. As all the downloaders,
`~skrub.datasets.fetch_employee_salaries` returns a dataset with attributes `X`, and `y`. `X` is a dataframe which contains
the features (aka design matrix, explanatory variables, independent variables). `y` is a column (pandas Series) which
contains the target (aka dependent, response variable) that we want to learn to predict from `X`. In this case `y` is the annual salary.

In [8]:
# colab only: uncomment to install skrub
! pip install skrub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 13.3 MB/s eta 0:00:00


In [10]:
from skrub.datasets import fetch_employee_salaries

dataset = fetch_employee_salaries()
employees, salaries = dataset.X, dataset.y
employees

,gender,department,department_name,division,assignment_category,employee_position_title,date_first_hired,year_first_hired
0,F,POL,Department of Police,MSB Information Mgmt and Tech Division Records...,Fulltime-Regular,Office Services Coordinator,09/22/1986,1986
1,M,POL,Department of Police,ISB Major Crimes Division Fugitive Section,Fulltime-Regular,Master Police Officer,09/12/1988,1988
2,F,HHS,Department of Health and Human Services,Adult Protective and Case Management Services,Fulltime-Regular,Social Worker IV,11/19/1989,1989
3,M,COR,Correction and Rehabilitation,PRRS Facility and Security,Fulltime-Regular,Resident Supervisor II,05/05/2014,2014
4,M,HCA,Department of Housing and Community Affairs,Affordable Housing Programs,Fulltime-Regular,Planning Specialist III,03/05/2007,2007
...,...,...,...,...,...,...,...,...
9223,F,HHS,Department of Health and Human Services,School Based Health Centers,Fulltime-Regular,Community Health Nurse II,11/03/2015,2015
9224,F,FRS,Fire and Rescue Services,Human Resources Division,Fulltime-Regular,Fire/Rescue Division Chief,11/28/1988,1988
9225,M,HHS,Department of Health and Human Services,Child and Adolescent Mental Health Clinic Serv...,Parttime-Regular,Medical Doctor IV - Psychiatrist,04/30/2001,2001
9226,M,CCL,County Council,Council Central Staff,Fulltime-Regular,Manager II,09/05/2006,2006


Most machine-learning algorithms work with arrays of numbers. The challenge here is that the employees dataframe is a
heterogeneous set of columns: some are numerical (`'year_first_hired'`), some dates (`'date_first_hired'`), some have a few
categorical entries (`'gender'`), some many (`'employee_position_title'`). Therefore our table needs to be "vectorized": processed
to extract numeric features.

`skrub` provides a custom transformer, called a `TableVectorizer` to preprocess the data for us.

In [11]:
from skrub import TableVectorizer

vectorizer = TableVectorizer()
vectorized_employees = vectorizer.fit_transform(employees)
vectorized_employees

,gender_F,gender_M,gender_nan,department_BOA,department_BOE,department_CAT,department_CCL,department_CEC,department_CEX,department_COR,...,employee_position_title_25,employee_position_title_26,employee_position_title_27,employee_position_title_28,employee_position_title_29,date_first_hired_year,date_first_hired_month,date_first_hired_day,date_first_hired_total_seconds,year_first_hired
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.028743,0.044707,-0.096881,-0.095238,0.033066,1986.0,9.0,22.0,5.277312e+08,1986.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.660438,0.196673,0.073715,0.078349,0.097700,1988.0,9.0,12.0,5.900256e+08,1988.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.061007,0.033798,-0.041610,0.002187,0.112171,1989.0,11.0,19.0,6.274368e+08,1989.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.073816,-0.235514,-0.319940,0.589514,0.021729,2014.0,5.0,5.0,1.399248e+09,2014.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.134185,0.167790,0.160017,0.183905,-0.009049,2007.0,3.0,5.0,1.173053e+09,2007.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9223,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.009527,0.210777,-0.177279,-0.011892,-0.035262,2015.0,11.0,3.0,1.446509e+09,2015.0
9224,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.010239,-0.039243,-0.032934,0.024126,-0.001734,1988.0,11.0,28.0,5.966784e+08,1988.0
9225,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.020786,0.008788,-0.022863,-0.030918,0.018792,2001.0,4.0,30.0,9.885888e+08,2001.0
9226,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.050039,0.072177,-0.020860,-0.020917,-0.065652,2006.0,9.0,5.0,1.157414e+09,2006.0


## A simple Pipeline for tabular data

The `TableVectorizer` outputs data that can be understood by a scikit-learn estimator. Therefore we can easily build a
2-step scikit-learn `Pipeline` that we can fit, test or cross-validate and that works well on tabular data.

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
import numpy as np

In [13]:
pipeline = make_pipeline(TableVectorizer(), RandomForestRegressor())
pipeline

Pipeline(steps=[('tablevectorizer', TableVectorizer()),
                ('randomforestregressor', RandomForestRegressor())])

**Note:** In colab this will take ~ 3-4min to run - you might want to skip this cell.

In [14]:
%%time

results = cross_validate(pipeline, employees, salaries)
scores = results["test_score"]
print(f"R2 score:  mean: {np.mean(scores):.3f}; std: {np.std(scores):.3f}")
print(f"mean fit time: {np.mean(results['fit_time']):.3f} seconds")

R2 score:  mean: 0.901; std: 0.018
mean fit time: 33.602 seconds
CPU times: user 2min 45s, sys: 136 ms, total: 2min 45s
Wall time: 2min 49s


Now let's swap out `scikit-learn`'s `RandomForestRegressor` for `cuML`'s and notice that it runs in seconds instead of
minutes.

In [15]:
from cuml.ensemble import RandomForestRegressor as cuRFR

pipeline = make_pipeline(TableVectorizer(), cuRFR())

In [16]:
%%time

results = cross_validate(pipeline, employees, salaries)
scores = results["test_score"]
print(f"R2 score:  mean: {np.mean(scores):.3f}; std: {np.std(scores):.3f}")
print(f"mean fit time: {np.mean(results['fit_time']):.3f} seconds")

R2 score:  mean: 0.895; std: 0.016
mean fit time: 3.046 seconds
CPU times: user 23.5 s, sys: 4.41 s, total: 27.9 s
Wall time: 21.5 s


## Conclusion

In this notebook, we learned:

* How to use cuML's drop-in replacements for scikit-learn estimators
* How cuML can accelerate machine learning workflows on GPU
* How to integrate cuML with scikit-learn Pipelines
* The performance benefits of GPU-accelerated machine learning with cuML

To learn more, we encourage you to visit the [cuML documentation](https://docs.rapids.ai/api/cuml/stable/)
